<a href="https://colab.research.google.com/github/ShahJahanBrohii/ML-Internship-Flyrank/blob/main/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

This is a ranking problem: the practical question is which keyword articles should be reviewed first. I compare Logistic Regression, a shallow Decision Tree, and a constrained Random Forest. Logistic Regression is the readable reference model; the tree models test whether simple nonlinearities add useful ranking signal. I select by held-out `precision@50`, because a review queue has limited capacity. All results are directional decision support on an observed label, not a causal claim.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
DATA_CANDIDATES = [
    Path("/content/content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"),
]
data_path = next((path for path in DATA_CANDIDATES if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

data = pd.read_csv(data_path)
lane = data.loc[data["content_type"].eq("keyword article")].copy()
lane["is_declining_label"] = lane["trend_direction"].astype(str).str.lower().eq("down").astype(int)

# Keep the model at the decision-time boundary: IDs, labels, and label-derived trend fields are excluded.
forbidden_features = {
    "content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label",
}
feature_candidates = [
    "days_since_last_update", "content_age_days", "word_count", "char_count",
    "search_volume", "competition", "cpc", "impressions_90d", "clicks_90d",
    "pageviews_90d", "sessions_90d", "users_90d", "engaged_sessions_90d",
    "ai_sessions_90d", "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d", "ctr",
    "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
feature_columns = [column for column in feature_candidates if column in lane.columns and column not in forbidden_features]
X = lane[feature_columns].apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
y = lane["is_declining_label"].astype(int)

assert not forbidden_features.intersection(X.columns)
assert y.nunique() == 2
print(f"Loaded {len(data):,} rows; keyword article lane: {len(lane):,} rows")
print(f"Target rate (observed declining label): {y.mean():.3f}; numeric features: {len(feature_columns)}")
print(f"Excluded from features: {sorted(forbidden_features)}")
display(lane[["is_declining_label", "impressions_90d", "days_since_last_update"]].describe().round(2))

Loaded 30,000 rows; keyword article lane: 27,207 rows
Target rate (observed declining label): 0.561; numeric features: 28
Excluded from features: ['client_id', 'content_id', 'is_declining_label', 'trend_direction', 'trend_pct']


,is_declining_label,impressions_90d,days_since_last_update
count,27207.00,27207.00,27207.00
mean,0.56,5712.94,48.29
std,0.50,17599.01,42.78
min,0.00,1.00,1.00
25%,0.00,152.00,20.00
50%,1.00,955.00,22.00
75%,1.00,4233.50,104.00
max,1.00,517715.00,373.00


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
# Hold out complete clients so content from the same client cannot appear in both train and test.
all_indices = np.arange(len(lane))
clients = lane["client_id"].fillna("unknown").astype(str)
unique_clients = clients.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.20)))
test_clients = set(shuffled_clients[:test_client_count])
test_mask = clients.isin(test_clients).to_numpy()
train_indices = all_indices[~test_mask]
test_indices = all_indices[test_mask]

# The fallback preserves a usable two-class test set if a future export has unusual client labels.
if (
    len(train_indices) == 0
    or len(test_indices) == 0
    or y.iloc[train_indices].nunique() < 2
    or y.iloc[test_indices].nunique() < 2
):
    train_indices, test_indices = train_test_split(
        all_indices, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )
    split_strategy = "stratified_row_holdout_fallback"
else:
    split_strategy = "client_holdout"

X_train, X_test = X.iloc[train_indices], X.iloc[test_indices]
y_train, y_test = y.iloc[train_indices], y.iloc[test_indices]
print(f"Split: {split_strategy}; train={len(train_indices):,}, test={len(test_indices):,}")
print(f"Clients: train={clients.iloc[train_indices].nunique()}, test={clients.iloc[test_indices].nunique()}, overlap={len(set(clients.iloc[train_indices]) & set(clients.iloc[test_indices]))}")
print(f"Test decline rate: {y_test.mean():.3f}")
assert set(clients.iloc[train_indices]).isdisjoint(set(clients.iloc[test_indices])) or split_strategy.endswith("fallback")

Split: client_holdout; train=25,776, test=1,431
Clients: train=25, test=6, overlap=0
Test decline rate: 0.460


## 3. Train + compare vs my baseline

The Week-4 baseline is rebuilt here on the same keyword-article rows and evaluated only on this test split. Its score is `impressions_90d` when a page is both stale (`days_since_last_update >= 180`) and visible (`impressions_90d >= 500`), otherwise zero. The model and baseline are compared with the same ranking metrics.

In [3]:
def precision_at_k(y_true, scores, k):
    evaluation = pd.DataFrame({"label": np.asarray(y_true), "score": np.asarray(scores)})
    top = evaluation.sort_values("score", ascending=False).head(min(k, len(evaluation)))
    return float(top["label"].mean()) if len(top) else 0.0

baseline_scores = np.where(
    (lane.iloc[test_indices]["days_since_last_update"].to_numpy() >= 180)
    & (lane.iloc[test_indices]["impressions_90d"].to_numpy() >= 500),
    lane.iloc[test_indices]["impressions_90d"].to_numpy(),
    0.0,
)

models = {
    "logistic_regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "decision_tree": DecisionTreeClassifier(
        class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE
    ),
    "random_forest": RandomForestClassifier(
        class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
        n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE
    ),
}

def metrics_for(scores, labels):
    predictions = (scores >= 0.5).astype(int)
    return {
        "ROC AUC": roc_auc_score(labels, scores),
        "Average precision": average_precision_score(labels, scores),
        "Precision@20": precision_at_k(labels, scores, 20),
        "Precision@50": precision_at_k(labels, scores, 50),
        "Recall": recall_score(labels, predictions, zero_division=0),
        "F1": f1_score(labels, predictions, zero_division=0),
        "Accuracy": accuracy_score(labels, predictions),
    }

results = {"baseline_week4_rule": metrics_for(baseline_scores, y_test.to_numpy())}
for name, model in models.items():
    model.fit(X_train, y_train)
    scores = model.predict_proba(X_test)[:, 1]
    results[name] = metrics_for(scores, y_test.to_numpy())

comparison = pd.DataFrame(results).T.round(3)
comparison.index.name = "approach"
display(comparison)

best_model_name = max(
    models,
    key=lambda name: (results[name]["Precision@50"], results[name]["Average precision"], results[name]["ROC AUC"]),
)
best_model = models[best_model_name]
best_test_scores = best_model.predict_proba(X_test)[:, 1]
print(f"Selected model: {best_model_name} (highest held-out Precision@50; seed={RANDOM_STATE})")
print(f"Base rate in test set: {y_test.mean():.3f}")

,ROC AUC,Average precision,Precision@20,Precision@50,Recall,F1,Accuracy
approach,,,,,,,
baseline_week4_rule,0.500,0.460,0.6,0.58,0.000,0.000,0.540
logistic_regression,0.617,0.615,1.0,1.00,0.261,0.371,0.593
decision_tree,0.775,0.664,1.0,0.86,0.985,0.775,0.737
random_forest,0.905,0.884,1.0,1.00,0.936,0.798,0.783


Selected model: random_forest (highest held-out Precision@50; seed=42)
Base rate in test set: 0.460


## 4. Errors and interpretation

The score is only useful if we can describe its mistakes. I inspect the selected model's false positives and false negatives, summarize errors by client-held-out test rows, and use permutation importance on the test set. Importance is directional: it shows which observed fields matter to this fitted model, not which fields cause decline.

In [4]:
test_view = lane.iloc[test_indices].copy()
test_view["model_score"] = best_test_scores
test_view["prediction"] = (best_test_scores >= 0.5).astype(int)
test_view["actual"] = y_test.to_numpy()
test_view["error_type"] = np.select(
    [test_view["prediction"].eq(1) & test_view["actual"].eq(0), test_view["prediction"].eq(0) & test_view["actual"].eq(1)],
    ["false_positive", "false_negative"],
    default="correct",
)

error_summary = (
    test_view.groupby("error_type")
    .agg(rows=("error_type", "size"), mean_score=("model_score", "mean"), decline_rate=("actual", "mean"))
    .reset_index()
)
print("Error summary (identifiers intentionally omitted):")
display(error_summary.round(3))

wrong = test_view.loc[test_view["error_type"].ne("correct")].copy()
wrong_display = wrong[[
    "error_type", "actual", "prediction", "model_score", "impressions_90d",
    "days_since_last_update", "avg_position", "word_count", "clicks_90d",
]].head(3).reset_index(drop=True)
print("Three representative wrong cases, with only decision-time fields:")
display(wrong_display.round(3))

permutation = permutation_importance(
    best_model, X_test, y_test, scoring="average_precision", n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1
)
importance = pd.DataFrame({"feature": X_test.columns, "importance": permutation.importances_mean})
importance = importance.sort_values("importance", ascending=False).head(10)
print("Top permutation-importance features (average precision decrease when shuffled):")
display(importance.round(4))

print(
    f"The selected model made {len(wrong):,} errors in the held-out test rows. "
    "The examples show why an observed decline proxy is hard: exposure, freshness, position, and depth can overlap without uniquely identifying the label."
)

Error summary (identifiers intentionally omitted):


,error_type,rows,mean_score,decline_rate
0,correct,1120,0.455,0.55
1,false_negative,42,0.461,1.00
2,false_positive,269,0.603,0.00


Three representative wrong cases, with only decision-time fields:


,error_type,actual,prediction,model_score,impressions_90d,days_since_last_update,avg_position,word_count,clicks_90d
0,false_positive,0,1,0.657,307,103,39.8,1342.0,0
1,false_negative,1,0,0.435,9,8,10.1,1585.0,0
2,false_positive,0,1,0.536,416,104,12.9,3393.0,3


Top permutation-importance features (average precision decrease when shuffled):


,feature,importance
17,impressions_last_30d,0.3127
20,impressions_prev_30d,0.2919
24,avg_position,0.0192
23,ctr,0.0047
8,clicks_90d,0.0018
19,sessions_last_30d,0.0016
10,sessions_90d,0.0010
5,competition,0.0008
25,engagement_rate,0.0004
21,clicks_prev_30d,0.0002


The selected model made 311 errors in the held-out test rows. The examples show why an observed decline proxy is hard: exposure, freshness, position, and depth can overlap without uniquely identifying the label.


## Self-check

- [x] Method choice explains why ranking models fit the lane and why `precision@50` selects the model.
- [x] The split is client-held-out when possible, with a reproducible seed and a guarded fallback.
- [x] The Week-4 rule baseline and learned models use the same held-out rows and metrics.
- [x] Errors, three representative wrong cases, and permutation importance are shown without identifiers.
- [x] Claims are directional and based on an observed label; trend-derived fields are excluded from features.
- [ ] Run every cell top to bottom, inspect the comparison table, and commit this notebook under `work/notebooks/`.